# Customer Churn Prediction System

This project is a Machine Learning based Customer Churn Prediction System developed using Python, Scikit-learn, and Streamlit. The main objective of the project is to predict whether a telecom customer is likely to churn or not based on customer-related information.

The dataset used in this project is the Telco Customer Churn dataset. Data preprocessing techniques were applied including:

* Removing unnecessary and leakage columns
* Handling missing values
* Converting data types
* Encoding categorical variables
* Feature scaling

The project uses two machine learning models:

1. Random Forest Classifier
2. Logistic Regression

A complete preprocessing and training pipeline was created using:

* Pipeline
* ColumnTransformer
* OneHotEncoder
* StandardScaler

Hyperparameter tuning was performed using GridSearchCV to improve model performance. The Random Forest model achieved better performance and was selected as the final model.

Model evaluation was performed using:

* Accuracy Score
* F1 Score
* Classification Report
* Confusion Matrix

The trained model was saved using Joblib for deployment purposes.

A user-friendly web application was developed using Streamlit where users can enter customer information such as:

* Gender
* Senior Citizen
* Tenure Months
* Monthly Charges
* Total Charges
* Contract Type
* Internet Service
* Payment Method

The application predicts whether the customer is likely to churn or not in real time.

For deployment and public access, Streamlit was connected with LocalTunnel to generate a public URL that allows users to access the application online.

Technologies Used:

* Python
* Pandas
* NumPy
* Scikit-learn
* Matplotlib
* Seaborn
* Streamlit
* Joblib
* LocalTunnel

This project demonstrates practical implementation of:

* Data preprocessing
* Machine learning pipelines
* Model optimization
* Model deployment
* Web application integration


In [ ]:
# =========================================
# CELL 1 — INSTALL LIBRARIES
# =========================================

!pip install pandas numpy scikit-learn joblib streamlit openpyxl seaborn matplotlib -q

In [ ]:
# =========================================
# CELL 2 — IMPORT LIBRARIES
# =========================================

import pandas as pd
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    confusion_matrix
)

In [ ]:
# =========================================
# CELL 3 — LOAD DATASET
# =========================================

df = pd.read_excel(
    "/content/drive/MyDrive/archive/Telco_customer_churn.xlsx"
)

df.head()

In [ ]:
# =========================================
# CELL 4 — DATA CLEANING
# =========================================

leakage_cols = [
    "Churn Value",
    "Churn Score",
    "Churn Reason",
    "Churn Category",
    "Customer Status"
]

df.drop(columns=leakage_cols, inplace=True, errors="ignore")

df.drop("CustomerID", axis=1, inplace=True)

df["Total Charges"] = pd.to_numeric(
    df["Total Charges"],
    errors="coerce"
)

df["Churn Label"] = df["Churn Label"].map({
    "Yes": 1,
    "No": 0
})

In [ ]:
# =========================================
# CELL 5 — FEATURES & TARGET
# =========================================

selected_cols = [
    "Gender",
    "Senior Citizen",
    "Tenure Months",
    "Monthly Charges",
    "Total Charges",
    "Contract",
    "Internet Service",
    "Payment Method"
]

X = df[selected_cols]

y = df["Churn Label"]

In [ ]:
# =========================================
# CELL 6 — COLUMN TYPES
# =========================================

categorical_cols = X.select_dtypes(
    include=["object"]
).columns

numerical_cols = X.select_dtypes(
    exclude=["object"]
).columns

print(categorical_cols)
print(numerical_cols)

In [ ]:
# =========================================
# CELL 7 — PREPROCESSING PIPELINES
# =========================================

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, numerical_cols),
    ("cat", cat_pipeline, categorical_cols)
])

In [ ]:
# =========================================
# CELL 8 — RANDOM FOREST PIPELINE
# =========================================

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

params = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [5, 10, None],
    "classifier__min_samples_split": [2, 5]
}

In [ ]:
# =========================================
# CELL 9 — TRAIN TEST SPLIT
# =========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# =========================================
# CELL 10 — GRID SEARCH
# =========================================

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=params,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

print("Best Parameters:")
print(grid_search.best_params_)

In [ ]:
# =========================================
# CELL 11 — MODEL EVALUATION
# =========================================

y_pred = best_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\nAccuracy Score:", accuracy)
print("F1 Score:", f1)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

In [ ]:
# =========================================
# CELL 12 — CONFUSION MATRIX
# =========================================

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Customer Churn Confusion Matrix")

plt.show()

In [ ]:
# =========================================
# CELL 13 — SAVE MODEL
# =========================================

os.makedirs("models", exist_ok=True)

joblib.dump(
    best_model,
    "models/best_pipeline.pkl"
)

print("MODEL SAVED SUCCESSFULLY")

In [ ]:
# =========================================
# CELL 14 — CREATE app.py
# =========================================

app_code = """
import streamlit as st
import pandas as pd
import joblib

model = joblib.load("models/best_pipeline.pkl")

st.title("Customer Churn Prediction")

gender = st.selectbox(
    "Gender",
    ["Male", "Female"]
)

senior = st.selectbox(
    "Senior Citizen",
    [0, 1]
)

tenure = st.number_input(
    "Tenure Months",
    0,
    100,
    12
)

monthly = st.number_input(
    "Monthly Charges",
    0.0,
    500.0,
    70.0
)

total = st.number_input(
    "Total Charges",
    0.0,
    10000.0,
    1000.0
)

contract = st.selectbox(
    "Contract",
    ["Month-to-month", "One year", "Two year"]
)

internet = st.selectbox(
    "Internet Service",
    ["DSL", "Fiber optic", "No"]
)

payment = st.selectbox(
    "Payment Method",
    [
        "Electronic check",
        "Mailed check",
        "Bank transfer (automatic)",
        "Credit card (automatic)"
    ]
)

input_df = pd.DataFrame({
    "Gender": [gender],
    "Senior Citizen": [senior],
    "Tenure Months": [tenure],
    "Monthly Charges": [monthly],
    "Total Charges": [total],
    "Contract": [contract],
    "Internet Service": [internet],
    "Payment Method": [payment]
})

if st.button("Predict"):

    prediction = model.predict(input_df)[0]

    if prediction == 1:
        st.error("Customer likely to churn")
    else:
        st.success("Customer not likely to churn")
"""

with open("app.py", "w") as f:
    f.write(app_code)

print("app.py created successfully")

In [ ]:
# =========================================
# CELL 15 — INSTALL LOCALTUNNEL
# =========================================

!npm install -g localtunnel

In [ ]:
# =========================================
# CELL 16 — RUN STREAMLIT
# =========================================

!streamlit run app.py &>/content/logs.txt &

In [ ]:
# =========================================
# CELL 17 — GENERATE PUBLIC URL
# =========================================

!npx localtunnel --port 8501

In [ ]:
from google.colab import files
files.download("app.py")